In [89]:
import os
import numpy as np
import pandas as pd
try:
    from thefuzz import process
except ImportError:
    from fuzzywuzzy import process

In [90]:
#all opten data names:
# Read in data
opten_path = '/mnt/common-hdd/raw-sources/opten-data/mta_cegalap.csv'
loc_path = '/mnt/common-hdd/raw-sources/opten-data/opten_budapest_here.csv'
beszamolo_path = '/mnt/common-hdd/raw-sources/opten-data/mta_beszamolo.csv'
teaor_path = '/mnt/common-ssd/zadorzsofi/telekom/Breathing_city/data/teaor08_struktura_2018_09_01.xls'

# Read the CSV file into a DataFrame
df = pd.read_csv(opten_path, encoding='latin-1', sep=';', dtype={'fotev_kod': str})
opten_loc = pd.read_csv(loc_path, dtype={'irsz': np.float64})
beszamolo = pd.read_csv(beszamolo_path, encoding='latin-1', sep=';')
teaor = pd.read_excel(teaor_path, header=1)

/tmp/ipykernel_740020/3788249242.py:9: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(opten_path, encoding='latin-1', sep=';', dtype={'fotev_kod': str})
/tmp/ipykernel_740020/3788249242.py:10: DtypeWarning: Columns (39,40) have mixed types. Specify dtype option on import or set low_memory=False.
  opten_loc = pd.read_csv(loc_path, dtype={'irsz': np.float64})
/tmp/ipykernel_740020/3788249242.py:11: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  beszamolo = pd.read_csv(beszamolo_path, encoding='latin-1', sep=';')


In [91]:
print(df.oo_cegj_sz.nunique())

# Filter df to hely == 'Budapest' and only keep rows where allapot is 'Mûködik'
df = df[(df['hely'] == 'Budapest') & (df['allapot'] == 'Mûködik')].copy()
print(df.oo_cegj_sz.nunique())

632384
180105


In [92]:
opten_loc2 = opten_loc[['h3_10', 'irsz', 'hely', 'teru', 'terj', 'hsz', 'eplt', 'lhaz']]

In [93]:
#merge df with opten_loc
merge_columns = ['irsz', 'hely', 'teru', 'terj', 'hsz', 'eplt', 'lhaz']
opten_loc_h3 = df.merge(opten_loc2, on=merge_columns, how='left')
#dropping rows where matching was not successful, most of them have non-existent tax numbers
opten_loc_h3 = opten_loc_h3.dropna(subset=['h3_10']).reset_index(drop=True)

In [94]:
# add information about income from beszamolo
besz_ev = beszamolo[beszamolo['ev'].isin([2018, 2019])]
besz_filt = besz_ev[['oo_cegj_sz', 'ev', 'arbev', 'brutto_hozzaadott_ertek']]

# Pivot the DataFrame to flatten and expand the data
flat_besz = besz_filt.pivot_table(index='oo_cegj_sz', columns='ev', 
                                 values=['arbev', 'brutto_hozzaadott_ertek'], 
                                 aggfunc='first')

# Reset index to move 'oo_cegj_sz' back to a regular column
flat_besz = flat_besz.reset_index()

# Rename columns with suffixes _2018 and _2019
flat_besz.columns = [f'{col}_{year}' if col != 'oo_cegj_sz' else 'oo_cegj_sz' for col, year in flat_besz.columns]

In [95]:
# merge flat_besz with opten_loc_h3
opten_infos = opten_loc_h3.merge(flat_besz, on='oo_cegj_sz', how='left')

In [96]:
# Checkpoint save: company + location + income data, before TEAOR info is added below
opten_infos.to_pickle('../data/data_gen/opten_ceginfo_ar.pkl')

## Adding TEÁOR Information

In [ ]:
# Match opten's fotev_kod to the official TEAOR'08 class codes.

# Primary match: by code number, against teaor08's 4-digit class-level codes
# A small number of codes aren't present in teaor08  these are matched by fuzzy name matching instead.

In [98]:
# Clean fotev_kod
opten_code = opten_infos[['fotev_kod', 'fotev_nev']].drop_duplicates().copy()
opten_code['fotev_kod'] = opten_code['fotev_kod'].str.strip()

ends_in_extraneous_4 = (opten_code['fotev_kod'].str.len() == 5) & (opten_code['fotev_kod'].str.endswith('4'))

#these will be fuzzy matched
ends_in_5char_3 = (opten_code['fotev_kod'].str.len() == 5) & (opten_code['fotev_kod'].str.endswith('3'))

opten_code['teaor_kod_clean'] = opten_code['fotev_kod'].where(
    ~ends_in_extraneous_4,
    opten_code['fotev_kod'].str[:4]
)

In [ ]:
teaor['Kód'] = teaor['Kód'].astype(str).str.strip()
teaor_class = teaor[
    (teaor['Kód'].str.len() == 4) & (teaor['Kód'].str.isdigit())
][['Kód', 'Megnevezés']].drop_duplicates()

In [100]:
# Primary match: by code number
opten_code2 = opten_code.merge(
    teaor_class.rename(columns={'Megnevezés': 'megnevezes_by_code'}),
    left_on='teaor_kod_clean', right_on='Kód', how='left'
)
opten_code2.drop(columns=['Kód'], inplace=True)

In [101]:
# Fallback: fuzzy-match the name for codes that didn't match numerically
unmatched = opten_code2.loc[opten_code2['megnevezes_by_code'].isna(), ['fotev_nev']].drop_duplicates()
unmatched['fotev_nev'] = unmatched['fotev_nev'].astype(str)
#after inspecting, the first to rows are incorrect
unmatched = unmatched.iloc[2:].reset_index(drop=True)

In [102]:
# Get the list of choices from teaor_class['Megnevezés'] and fuzzy-match each unmatched name
choices = teaor_class['Megnevezés'].tolist()

unmatched['megnevezes_resolved'] = unmatched['fotev_nev'].apply(
    lambda name: process.extractOne(name, choices)[0]
)

In [103]:
# Merge the resolved names back and consolidate into a single Megnevezés column
opten_code2 = opten_code2.merge(unmatched[['fotev_nev', 'megnevezes_resolved']], on='fotev_nev', how='left')
opten_code2['Megnevezés'] = opten_code2['megnevezes_by_code'].combine_first(opten_code2['megnevezes_resolved'])
opten_code2.drop(columns=['megnevezes_by_code', 'megnevezes_resolved'], inplace=True)

In [104]:
opten_code2.head()

,fotev_kod,fotev_nev,teaor_kod_clean,Megnevezés
0,46904,Vegyestermékkörû nagykereskedelem,4690,Vegyestermékkörű nagykereskedelem
1,68204,"Saját tulajdonú, bérelt ingatlan bérbeadása, ü...",6820,"Saját tulajdonú, bérelt ingatlan bérbeadása, ü..."
2,13204,Textilszövés,1320,Textilszövés
3,69204,"Számviteli, könyvvizsgálói, adószakértõi tevék...",6920,"Számviteli, könyvvizsgálói, adószakértői tevék..."
4,49414,Közúti áruszállítás,4941,Közúti áruszállítás


In [105]:
# merge back with official teaor code
opten_code2 = opten_code2.merge(teaor_class, on='Megnevezés', how='left')
opten_code2.rename(columns={'Kód': 'teaor_kod'}, inplace=True)

In [106]:
#check if there are any unmatched
opten_code2[opten_code2['teaor_kod'].isna()]

#drop NaN and manually add Megnevezés and code Nyomás (kivéve: napilap) 1812

,fotev_kod,fotev_nev,teaor_kod_clean,Megnevezés,teaor_kod
274,NaN,NaN,NaN,NaN,NaN
279,22223,Máshova nem sorolt nyomás,22223,NaN,NaN
305,85954,NaN,8595,NaN,NaN
457,44754,NaN,4475,NaN,NaN
462,52504,NaN,5250,NaN,NaN
467,74994,NaN,7499,NaN,NaN
501,85214,NaN,8521,NaN,NaN
512,47404,NaN,4740,NaN,NaN
527,62904,NaN,6290,NaN,NaN
536,85994,NaN,8599,NaN,NaN


In [ ]:
mask = opten_code2['teaor_kod'].isna() & (opten_code2['fotev_nev'] == 'Máshova nem sorolt nyomás')
opten_code2.loc[mask, 'Megnevezés'] = 'Nyomás (kivéve: napilap)'
opten_code2.loc[mask, 'teaor_kod'] = '1812'

# Check what is still unmatched
opten_code2[opten_code2['teaor_kod'].isna()]

# Drop any remaining rows that are still NaN
opten_code2 = opten_code2.dropna(subset=['teaor_kod'])

In [108]:
opten_code2['teaor_2'] = opten_code2['teaor_kod'].astype(str).str[:2]

In [109]:
# Define the dictionary that maps each letter to its corresponding list of strings
mapping = {
    'A': ['01', '02', '03'],
    'B': ['05', '06', '07', '08', '09'],
    'C': [str(i) for i in range(10, 34)],
    'D': ['35'],
    'E': ['36', '37', '38', '39'],
    'F': ['41', '42', '43'],
    'G': ['45', '46', '47'],
    'H': ['49', '50', '51', '52', '53'],
    'I': ['55', '56'],
    'J': ['58', '59', '60', '61', '62', '63'],
    'K': ['64', '65', '66'],
    'L': ['68'],
    'M': [str(i) for i in range(69, 76)],
    'N': [str(i) for i in range(77, 83)],
    'O': ['84'],
    'P': ['85'],
    'Q': ['86', '87', '88'],
    'R': ['90', '91', '92', '93'],
    'S': ['94', '95', '96'],
    'T': ['97', '98'],
    'U': ['99']
}

# Create a function to map teaor_2 to the corresponding letter
def map_to_letter(teaor_2):
    for letter, values in mapping.items():
        if teaor_2 in values:
            return letter
    return None  # In case teaor_2 does not match any value

# Apply the function to create the new column
opten_code2['teaor_betu'] = opten_code2['teaor_2'].apply(map_to_letter)

In [110]:
# Finalize the crosswalk: drop the intermediate cleaned-code helper column and dedupe
opten_codes_to_teaor = opten_code2.drop(columns=['teaor_kod_clean']).drop_duplicates()

In [111]:
opten_infos = opten_infos.merge(opten_codes_to_teaor, on=['fotev_kod', 'fotev_nev'], how='left')
opten_infos2 = opten_infos.drop(columns=['fotev_kod', 'fotev_nev'])

In [112]:
opten_infos.head()

,oo_cegj_sz,allapot,adoszam_allapot,cegnev,adoszam,alapitas,megszunes,irsz,hely,helyresz,...,tulaj_lancban_kulfoldi,h3_10,arbev_2018,arbev_2019,brutto_hozzaadott_ertek_2018,brutto_hozzaadott_ertek_2019,Megnevezés,teaor_kod,teaor_2,teaor_betu
0,01 09 061282,Mûködik,Érvényes az adószám,DAGENT Kereskedelmi és Szolgáltató Korlátolt F...,10231890-2-41,1989.01.11.,NaN,1013.0,Budapest,NaN,...,1,8a1e03785047fff,151297.0,NaN,49981.0,NaN,Vegyestermékkörű nagykereskedelem,4690,46,G
1,01 09 061332,Mûködik,Érvényes az adószám,HEVIT Ipari Termelõ és Kereskedelmi Szolgáltat...,10229741-2-42,1989.02.01.,NaN,1105.0,Budapest,NaN,...,0,8a1e03632cd7fff,65613.0,72274.0,22435.0,33917.0,"Saját tulajdonú, bérelt ingatlan bérbeadása, ü...",6820,68,L
2,08 10 000224,Mûködik,Érvényes az adószám,Picatrix Zártkörûen Mûködõ Részvénytársaság,10219838-2-41,1988.10.12.,NaN,1016.0,Budapest,NaN,...,1,8a1e037852b7fff,0.0,NaN,-230448.0,NaN,Textilszövés,1320,13,C
3,01 09 061149,Mûködik,Érvényes az adószám,PIRAMID Építõipari és Szolgáltató Korlátolt Fe...,10224928-1-42,1989.01.02.,NaN,1143.0,Budapest,NaN,...,0,8a1e037ab847fff,5852.0,9076.0,6075.0,2271.0,"Számviteli, könyvvizsgálói, adószakértői tevék...",6920,69,M
4,11 09 001247,Mûködik,Érvényes az adószám,KALMÁR Szállítási és Szolgáltató Korlátolt Fel...,10418972-2-41,1990.10.01.,NaN,1034.0,Budapest,NaN,...,0,8a1e037a116ffff,189968.0,NaN,33244.0,NaN,Közúti áruszállítás,4941,49,H


In [121]:
# Merge the resolved TEAOR information onto the company-level dataset.
opten_infos2.to_pickle('../data/data_gen/opten_ceginfo_ar_teaor.pkl')

In [122]:
print(opten_infos2.oo_cegj_sz.nunique())
print(len(opten_infos2[opten_infos2['letszam_besz18'] >3 ]))

178374
29409


In [123]:
opten_large = opten_infos2[opten_infos2['letszam_besz18'] >3 ]
opten_large.to_pickle('../data/data_gen/opten_ceginfo_ar_teaor_large.pkl')

In [124]:
opten_large.columns

Index(['oo_cegj_sz', 'allapot', 'adoszam_allapot', 'cegnev', 'adoszam',
       'alapitas', 'megszunes', 'irsz', 'hely', 'helyresz', 'teru', 'terj',
       'hsz', 'eplt', 'lhaz', 'emelet', 'ajto', 'hrsz', 'cimid_rowid',
       'letszam_besz16', 'letszam_besz17', 'letszam_besz18', 'letszam_besz19',
       'letszam_nav_20_januar', 'letszam_nav_20_februar',
       'letszam_nav_20_marcius', 'letszam_nav_20_aprilis',
       'letszam_nav_20_majus', 'letszam_nav_20_junius', 'letszam_ksh',
       'tulaj_lancban_kulfoldi', 'h3_10', 'arbev_2018', 'arbev_2019',
       'brutto_hozzaadott_ertek_2018', 'brutto_hozzaadott_ertek_2019',
       'Megnevezés', 'teaor_kod', 'teaor_2', 'teaor_betu'],
      dtype='object')